### Import Dependencise

In [40]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams,Distance,PayloadSchemaType,PointStruct,SparseVectorParams,Document,Prefetch,FusionQuery
from qdrant_client import models
import pandas as pd
from openai import OpenAI
import fastembed
import os

In [41]:
client=OpenAI()

In [3]:
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")

### Create Quadrant Collection For Hybrid Collection

In [4]:
quadrant_client=QdrantClient(url="http://localhost:6333/")

In [35]:
quadrant_client.create_collection(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    vectors_config={
        "text-embedding-model-3-small":VectorParams(size=1536,distance=Distance.COSINE)
    }
    ,
    sparse_vectors_config={
        "bm25":SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [36]:
def get_embedding(text,model="text-embedding-3-small"):
    response=openai.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

In [5]:
def get_embedding_batch(text_list,model="text-embedding-3-small",batch_size=100):
    if len(text_list)<=batch_size:
        response=openai.embeddings.create(input=text_list,model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings=[]
    counter=1
    for i in range(0,len(text_list),batch_size):
        batch=text_list[i:i+batch_size]
        response=openai.embeddings.create(input=batch,model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        counter+=1
    
    return all_embeddings
        

### Preprocesse Amazon data

In [6]:
df_items=pd.read_json("C:\\Users\\jaysi\\Desktop\\Desktop\\Ai-engineering\\data\\meta_Electronics_2022_23_with_categeory_rating_100_sample_2000.jsonl",lines=True)

In [7]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[],169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...",[],24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN


In [8]:
def preprocessed_decription(row):
    return f"{row['title']}{''.join(row['features'])}"

In [9]:
def extrect_first_large_image(row):
    return row['images'][0].get("large","")

In [10]:
df_items['description']=df_items.apply(preprocessed_decription,axis=1)
df_items['image']=df_items.apply(extrect_first_large_image,axis=1)

In [11]:
data_to_embed=df_items[["description","image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [12]:
len(data_to_embed)

2000

In [13]:
text_to_embed=[data["description"] for data in data_to_embed]

In [ ]:
text_to_embed

In [15]:
embeddings = get_embedding_batch(text_to_embed)


In [ ]:
print(embeddings)

In [17]:
pointstructs=[]
i=1
for embedding,data in zip(embeddings,data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-model-3-small":embedding,
                "bm25":Document(
                    text=data["description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i+=1

In [ ]:
pointstructs

In [23]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[0:500],
    wait=True
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [24]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[500:1000],
    wait=True
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [25]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[1000:1500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [34]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[1500:2000],
    wait=True
)

UpdateResult(operation_id=5, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrival

In [33]:
def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [43]:
def reteriver_data(query, quadrant_client, k):
    query_embedding = get_embedding(query)

    result = quadrant_client.query_points(
        collection_name="Amazon-items-collection-02-hybrid-serach",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-model-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )

        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )


In [44]:
result=reteriver_data("can i get some ipad",quadrant_client,k=20)

In [45]:
result

(['Case for iPad Pro 12.9 2022/2021/2020/2018 Gen 6th/5th/4th/3rd,2 in 1 Detachable Magnetic Case,Anti-Fingerprint Frosted Case and Washable Leather Protective Cover Support Pen Charger(Pink)✅【Protective Case for iPad Pro 12.9】❗❗No Pen included!❗❗The protective cover is designed for iPad Pro 12.9" 6th Gen 2022 (Model number: A2764/A2436/A2437/A2766),5th Gen 2021 (Model number: A2378/A2379/A2461/A2462),4th Gen 2020 (A2229/A2069/A2232/A2233),3rd Gen 2018 (A1876/A1895/A1983/A2014).♐It\'s Not compatible with any other devices. Please check the model number of your iPad back bottom.✅【2 in 1 Detachable & Magnetic Cover】The detachable design is more convenient to carry and use. When it\'s used as a protective case, you have a video mode or a writing mode. When it detached, you get the phone holder.✅【Washable & Stain-resistant Case】The premium leather and frosted case are both washable. You can easily get a new look on your case with a single tissue, even if it\'s stained with coffee, oil, or 